# 30 Jahre in die Zukunft

**Einstiegsfrage:** Zwei Personen sparen 40 Jahre lang genau gleich viel in genau denselben Fonds. Im Modell hat die eine am Ende rund zwölfmal so viel wie die andere. War sie klüger?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2027)


def pruefe(name, wert, soll, toleranz):
    """Selbstkontrolle: vergleicht Ihren Wert mit dem erwarteten Bereich."""
    if wert is None:
        print(f"{name}: noch nicht bearbeitet")
    elif abs(wert - soll) <= toleranz:
        print(f"{name}: passt ({wert:.4g})")
    else:
        print(f"{name}: prüfen Sie Ihren Code, erwartet wird etwa {soll:.4g}, Sie haben {wert:.4g}")

## Aufgabe 1: Viele Verläufe
Simulieren Sie 100'000 Verläufe über 30 Jahre: Wachstum 6 %, Volatilität 16 %, Start 1000 CHF.

*Hilfestellung:* `rng.normal(0.06, 0.16, size=(pfade, jahre))` erzeugt die Renditen. Summieren Sie pro Zeile und wenden Sie `np.exp` an.

In [ ]:
pfade, jahre = 100_000, 30
vermoegen = None
r = rng.normal(0.06, 0.16, size=(pfade, jahre))
vermoegen = 1000 * np.exp(r.sum(axis=1))
median = None if vermoegen is None else np.median(vermoegen)
pruefe('Median', median, 6050, 120)

## Aufgabe 2: Wie schlecht kann es laufen?
Bestimmen Sie das 5-%-Perzentil und den Anteil Verläufe unter 1000 CHF.

In [ ]:
p5 = verlust = None
p5 = np.percentile(vermoegen, 5)
verlust = np.mean(vermoegen < 1000)
pruefe('5-%-Perzentil', p5, 1430, 60)
pruefe('Verlustanteil', verlust, 0.02, 0.004)

## Challenge: Mittelwert gegen Median
Berechnen Sie den Mittelwert und erklären Sie, weshalb rund zwei Drittel aller Personen unter dem Durchschnitt liegen.

*Hilfestellung:* Für eine Lognormalverteilung gilt $E[X] = e^{\mu + \sigma^2/2}$, hier mit $\mu = 30 \cdot 0.06$ und $\sigma^2 = 30 \cdot 0.16^2$.

In [ ]:
anteil_unter = None
anteil_unter = np.mean(vermoegen < vermoegen.mean())
pruefe('Anteil unter dem Durchschnitt', anteil_unter, 0.669, 0.01)

## Challenge: Sparplan oder alles auf einmal?
Sie haben 12'000 CHF. Entweder legen Sie alles sofort an oder monatlich 100 CHF während 10 Jahren. Monatliches Wachstum 0.06/12, Monatsvolatilität $0.16/\sqrt{12}$. Wie oft gewinnt die Einmalanlage?

*Hilfestellung:* Die Rate im Monat $t$ wächst nur über die verbleibenden Monate. `np.cumsum(r[:, ::-1], axis=1)[:, ::-1]` liefert diese Restsummen.

In [ ]:
sieg_einmal = None
r = rng.normal(0.06 / 12, 0.16 / np.sqrt(12), size=(100_000, 120))
einmal = 12_000 * np.exp(r.sum(axis=1))
plan = 100 * np.exp(np.cumsum(r[:, ::-1], axis=1)[:, ::-1]).sum(axis=1)
sieg_einmal = np.mean(einmal > plan)
pruefe('Einmalanlage gewinnt', sieg_einmal, 0.836, 0.01)